# Documentación Proyecto Polaris — Proyecto de Bienestar

**Universidad de La Sabana**

_**Integrantes:** Samuel Rusinque, Fabiana Miranda, Vanesa Reyes, Nicolás Bejarano_

## Contexto de problema:

Dirección de Bienestar y Experiencia del Estudiante de la Sabana nos plantea bajo el ejemplo “Natalia”, un problema acerca de los servicios que la universidad le ofrece. A pesar de la visibilidad de estos, existen dos problemas:

El primero de estos es que la red de servicios para el estudiante de la universidad es “un mapa invisible”, ya que esta red existe, sin embargo, no está conectada entre sí y mucho menos actualizada.

El segundo problema es que, las bases de datos con la información de los estudiantes (Género, participaciones, promedio, etc.), están dispersas en varias bases de excel, es decir no existía una sola base de datos con toda la información, sino que los datos estaban separados.

## Implementación del Código

### Importar librerías
Se importan las siguientes librerías:

- **`re`**: permite trabajar con **expresiones regulares** para buscar, validar y reemplazar patrones de texto.
- **`unicodedata`**: se usa para **normalizar texto**, por ejemplo, eliminar tildes y otros caracteres acentuados.
- **`numpy` (`np`)**: facilita operaciones **numéricas** y manejo eficiente de arreglos.
- **`pandas` (`pd`)**: se utiliza para la **manipulación y análisis de datos** en tablas o DataFrames.
- **`matplotlib.pyplot` (`plt`)**: sirve para la **creación de gráficos** y visualizaciones básicas.
- **`seaborn` (`sns`)**: complementa a Matplotlib con gráficos más **estilizados y estadísticos**.




In [1]:
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────────────────────────────────────
# FLUJO DE TRABAJO
# ─────────────────────────────────────────────────────────────────────────────
# 1. Cargar datos desde archivos Excel (pandas.read_excel)
# 2. Limpiar: normalizar texto, eliminar duplicados, validar tipos de datos (re, unicodedata)
# 3. Transformar: crear variables derivadas, agrupar, fusionar tablas (pandas)
# 4. Analizar: correlaciones, regresión, clustering (numpy, scikit-learn)
# 5. Visualizar: gráficos exploratorios y resultados (matplotlib, seaborn)

In [2]:
# Instalar la librería openpyxl para leer archivos Excel
%pip install -q openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Cargar los datos desde los archivos Excel
df1 = pd.read_excel(
    r"Data\Consolidado - Información de Caracterización (2013 - 2025) (VE) (26.1).xlsx"
)
df2 = pd.read_excel(
    r"Data\Consolidado - Registros de Asistencia (2015-2022) (VE) (26.1).xlsx"
)
df3 = pd.read_excel(
    r"Data\Consolidado - Registros de Asistencias (2023 - 20251) (VE) (26.1).xlsx"
)

## 1. Comprensión general

Dimensión del problema, tipos de datos, muestra de filas, estadísticos de variables numéricas, cardinalidad en texto y observaciones iniciales **sobre los datos tal como vienen del archivo** (antes de la limpieza de la sección 2).

In [4]:
# Función para mostrar un resumen de comprensión de cada DataFrame

def resumen_comprension(df, nombre):
    """
    Genera un reporte completo de exploración de un DataFrame.
    
    Parámetros:
    -----------
    df : pandas.DataFrame
        DataFrame a analizar
    nombre : str
        Etiqueta descriptiva del dataset
    
    Muestra:
    --------
    - Dimensiones y memoria usada
    - Tipos de datos
    - Conteo de valores no nulos
    - Primeras 3 filas
    - Estadísticas descriptivas (variables numéricas)
    - Cardinalidad de texto (valores únicos)
    """
    print("\n" + "=" * 72)
    print(f" {nombre}")
    print("=" * 72)
    
    # Dimensiones y memoria
    print(f"Dimensiones (filas × columnas): {df.shape[0]:,} × {df.shape[1]}")
    mem_mb = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"Memoria aproximada (deep): {mem_mb:.2f} MiB\n")
    
    # Tipos de datos
    print("Tipos de datos por columna:")
    print(df.dtypes.to_string())
    
    # Valores no nulos
    print("\nConteo de valores no nulos por columna:")
    print(df.count().to_string())
    
    # Muestra de filas
    print("\nPrimeras filas:")
    print(df.head(3).to_string())
    
    # Estadísticas numéricas
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols):
        print("\nResumen estadístico (variables numéricas):")
        print(df[num_cols].describe().T.to_string())
    
    # Cardinalidad de texto
    obj_cols = df.select_dtypes(include="object").columns
    if len(obj_cols):
        print("\nCardinalidad — valores únicos en columnas de texto (orden descendente):")
        print(df[obj_cols].nunique(dropna=True).sort_values(ascending=False).to_string())

# Mostrar el resumen de comprensión para cada DataFrame
for etiqueta, d in [
    ("Dataset 1 — Caracterización (2013–2025)", df1),
    ("Dataset 2 — Asistencia (2015–2022)", df2),
    ("Dataset 3 — Asistencia (2023–2025)", df3),
]:
    resumen_comprension(d, etiqueta)


 Dataset 1 — Caracterización (2013–2025)
Dimensiones (filas × columnas): 212,263 × 16
Memoria aproximada (deep): 132.85 MiB

Tipos de datos por columna:
Ciclo lectivo                                  object
Código                                         object
Ciudad Dirección física                        object
SEXO                                           object
Organización académica                         object
Grupo Académico                                object
Programa Académico Principal                    int64
Descripción programa  Principal                object
Promedio acumulado Programa principal(PAM)    float64
Promedio semestral Programa principal(PCP)    float64
Ubicación semestral                            object
Doble programa                                 object
Estado Doble Programa                          object
Programa Académico Secundario                 float64
Descripción programa  Secundario               object
Créditos inscritos                  

## 2. Limpieza básica

- **Valores faltantes**: conteo y porcentaje por columna; se documentan; no se imputa sin criterio de negocio (los nulos estructurales se conservan).
- **Duplicados**: filas idénticas en todas las columnas; se eliminan conservando la primera ocurrencia.
- **Ruido en texto**: espacios sobrantes, mayúsculas y tildes inconsistentes; se unifica con normalización Unicode (NFKD, sin marcas combinantes) y nombres de columna estables (`snake_case`).
- **Inconsistencias numéricas**: conteo de notas (PAM/PCP) fuera del intervalo [0, 5] cuando aplique el nombre de columna.

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNCIONES DE LIMPIEZA Y DIAGNÓSTICO DE DATOS
# ─────────────────────────────────────────────────────────────────────────────
# Este módulo agrupa funciones reutilizables para normalizar, validar y reportar
# la calidad de DataFrames antes del análisis principal.

def limpiar_string(x, espacios_a_guion_bajo=False):
    """
    Normaliza y limpia un valor de texto: elimina acentos, espacios extremos 
    y convierte a minúsculas.
    
    Parámetros:
    -----------
    x : str o valor no-string
        Texto a limpiar
    espacios_a_guion_bajo : bool, default=False
        Si True, reemplaza espacios internos con guión bajo
    
    Retorna:
    --------
    str o valor original
        Texto normalizado en minúsculas sin acentos, o valor original si no es string
    
    Ejemplo:
    --------
    >>> limpiar_string("Héctor García")
    'hector garcia'
    >>> limpiar_string("Base de Datos", espacios_a_guion_bajo=True)
    'base_de_datos'
    """
    if not isinstance(x, str):
        return x
    # Normalización Unicode NFKD y eliminación de marcas combinantes
    nfkd = unicodedata.normalize("NFKD", x)
    sin_acentos = "".join(c for c in nfkd if unicodedata.category(c) != "Mn")
    s = sin_acentos.lower().strip()
    if espacios_a_guion_bajo:
        s = re.sub(r"\s+", "_", s)  # espacios → guión bajo
        s = re.sub(r"_+", "_", s).strip("_")  # guiones múltiples → uno
    return s


def limpiar_nombres_columnas(df):
    """
    Limpia los nombres de las columnas de un DataFrame aplicando normalización
    Unicode y conversión a snake_case.
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame con columnas a normalizar
    
    Retorna:
    --------
    pd.DataFrame
        Copia del DataFrame con nombres de columnas en snake_case sin acentos
    """
    out = df.copy()
    out.columns = [limpiar_string(c, espacios_a_guion_bajo=True) for c in out.columns]
    return out


def limpiar_df(df):
    """
    Aplica limpieza integral a un DataFrame:
    1. Normaliza nombres de columnas (snake_case, sin acentos)
    2. Limpia todas las celdas de texto (minúsculas, sin acentos, espacios extremos)
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame a limpiar
    
    Retorna:
    --------
    pd.DataFrame
        Copia del DataFrame con columnas y celdas de texto normalizadas
    
    Nota:
    -----
    Valores no-string y NaN se mantienen intactos.
    """
    out = limpiar_nombres_columnas(df)
    for col in out.select_dtypes(include="object").columns:
        out[col] = out[col].map(limpiar_string)
    return out


def tabla_faltantes(df):
    """
    Genera un reporte de valores faltantes (NaN) por columna.
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame a analizar
    
    Retorna:
    --------
    pd.DataFrame
        Tabla con columnas:
        - 'nulos': conteo de valores faltantes
        - 'pct': porcentaje relativo (redondeado a 2 decimales)
        Ordenado de mayor a menor cantidad de nulos; solo incluye columnas 
        que tienen al menos un valor faltante.
    """
    n = len(df)
    t = df.isnull().sum()
    p = (t / n * 100).round(2)
    tab = pd.DataFrame({"nulos": t, "pct": p})
    return tab[tab["nulos"] > 0].sort_values("nulos", ascending=False)


def conteo_duplicados_fila_completa(df):
    """
    Cuenta el número de filas duplicadas (todas las columnas idénticas) en el DataFrame.
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame a analizar
    
    Retorna:
    --------
    int
        Número de filas duplicadas (excluyendo la primera ocurrencia)
    """
    return int(df.duplicated().sum())


def celdas_texto_con_espacios_extremos(df):
    """
    Cuenta el número de celdas de texto que tienen espacios al inicio o final
    (indicador de necesidad de limpieza).
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame a analizar
    
    Retorna:
    --------
    int
        Total de celdas de texto con espacios al inicio/final en todo el DataFrame
    """
    total = 0
    for col in df.select_dtypes(include="object").columns:
        s = df[col].dropna()
        total += int(s.map(lambda x: isinstance(x, str) and x != x.strip()).sum())
    return total


def conteo_notas_fuera_rango(df, cmin=0.0, cmax=5.0):
    """
    Identifica valores numéricos que salen del rango esperado [cmin, cmax]
    en columnas que representen calificaciones (PAM, PCP).
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame a validar
    cmin : float, default=0.0
        Límite inferior del rango válido
    cmax : float, default=5.0
        Límite superior del rango válido
    
    Retorna:
    --------
    dict o str vacía
        Diccionario {columna: conteo} de anomalías encontradas;
        devuelve dict vacío {} si no hay inconsistencias
    
    Ejemplo:
    --------
    >>> conteo_notas_fuera_rango(df1)
    {'promedio_acumulado_programa_principal(pam)': 3}  # 3 valores fuera de [0,5]
    """
    filas = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        cl = col.lower()
        if "pam" in cl or "pcp" in cl:
            mask = df[col].notna() & ((df[col] < cmin) | (df[col] > cmax))
            n = int(mask.sum())
            if n:
                filas[col] = n
    return filas

In [6]:
# Diagnóstico previo a la normalización de los datos crudos.
# Este bloque inspecciona cada DataFrame antes de aplicar limpieza estructural:
# - valores faltantes por columna
# - filas duplicadas completas
# - celdas de texto con espacios al inicio o al final
# - valores numéricos fuera del rango esperado [0, 5] en columnas tipo PAM/PCP

print("Diagnóstico previo a normalizar (datos crudos)\n")

for nombre, df in [
    ("df1 — Caracterización", df1),
    ("df2 — Asistencia 2015–2022", df2),
    ("df3 — Asistencia 2023–2025", df3),
]:
    print("-" * 72)
    print(nombre)

    # Reporte de nulos
    print("Faltantes (columnas con al menos un nulo):")
    faltantes = tabla_faltantes(df)
    print(faltantes.to_string() if len(faltantes) else "  (ninguno)")

    # Duplicados completos
    duplicadas = conteo_duplicados_fila_completa(df)
    print(f"Filas duplicadas (todas las columnas): {duplicadas:,}")

    # Espacios al inicio/final en columnas de texto
    espacios = celdas_texto_con_espacios_extremos(df)
    print(f"Celdas de texto con espacios al inicio/fin: {espacios:,}")

    # Validación de rangos para columnas que parecen promedios académicos
    fuera_rango = conteo_notas_fuera_rango(df)
    print(
        "Notas numéricas fuera de [0, 5] (columnas PAM/PCP):",
        fuera_rango if fuera_rango else "ninguna detectada",
    )
    print()

Diagnóstico previo a normalizar (datos crudos)

------------------------------------------------------------------------
df1 — Caracterización
Faltantes (columnas con al menos un nulo):
                                             nulos    pct
Programa Académico Secundario               203105  95.69
Descripción programa  Secundario            203105  95.69
Estado Doble Programa                       203016  95.64
Promedio semestral Programa principal(PCP)   33442  15.75
Ciudad Dirección física                      13216   6.23
Ubicación semestral                           8439   3.98
Promedio acumulado Programa principal(PAM)    4968   2.34
Organización académica                           9   0.00
Filas duplicadas (todas las columnas): 0
Celdas de texto con espacios al inicio/fin: 0
Notas numéricas fuera de [0, 5] (columnas PAM/PCP): ninguna detectada

------------------------------------------------------------------------
df2 — Asistencia 2015–2022
Faltantes (columnas con al menos u

In [7]:
# Limpieza y eliminación de duplicados (documentado)
#
# Objetivo:
#  - Normalizar texto y nombres de columnas en cada DataFrame usando `limpiar_df`
#  - Detectar y eliminar filas duplicadas (todas las columnas iguales)
#  - Informar conteo de filas antes, número de duplicados eliminados y filas después
#  - Mostrar una muestra inicial (primeras filas) de df1 tras la limpieza
#
# Notas:
#  - `limpiar_df` ya está definida en una celda anterior y se encarga de:
#      * normalizar nombres de columnas a snake_case sin acentos
#      * limpiar celdas de texto (minúsculas, sin acentos, trim)
#  - La eliminación de duplicados se hace in-place con `drop_duplicates(inplace=True)`.

# Aplicar la limpieza textual/estructural a cada DataFrame
df1 = limpiar_df(df1)
df2 = limpiar_df(df2)
df3 = limpiar_df(df3)

# Para cada DataFrame: informar antes, eliminar duplicados y mostrar resultado
for dframe, nombre in [(df1, "df1"), (df2, "df2"), (df3, "df3")]:
    # filas antes de eliminar duplicados
    antes = len(dframe)
    # número de filas que pandas identifica como duplicadas (excluye la primera ocurrencia)
    n_dup = int(dframe.duplicated().sum())
    # eliminar filas duplicadas conservando la primer aparición
    dframe.drop_duplicates(inplace=True)
    # filas después de la eliminación
    despues = len(dframe)

    # salida resumida (formatos con separador de miles para legibilidad)
    print(
        f"{nombre}: filas antes {antes:,} | duplicadas eliminadas {n_dup:,} | filas después {despues:,}"
    )

# Mostrar una pequeña muestra de df1 para inspección visual tras la limpieza
print("\nTras limpieza — muestra df1 (primeras 3 filas):")
print(df1.head(3).to_string())

df1: filas antes 212,263 | duplicadas eliminadas 0 | filas después 212,263
df2: filas antes 508,692 | duplicadas eliminadas 288,353 | filas después 220,339
df3: filas antes 291,852 | duplicadas eliminadas 204,725 | filas después 87,127

Tras limpieza — muestra df1 (primeras 3 filas):
    ciclo_lectivo            codigo ciudad_direccion_fisica   sexo        organizacion_academica               grupo_academico  programa_academico_principal  descripcion_programa_principal  promedio_acumulado_programa_principal(pam)  promedio_semestral_programa_principal(pcp) ubicacion_semestral doble_programa estado_doble_programa  programa_academico_secundario descripcion_programa_secundario  creditos_inscritos
0  periodo 2015-1  c101489ho303414h             bogota d.c.  mujer                    psicologia                    psicologia                             8                      psicologia                                        4.33                                        4.29          semestre 7  

In [8]:
# Mostrar el conteo de valores nulos por columna para cada DataFrame

tf1 = tabla_faltantes(df1)
print("Faltantes después de limpiar (df1):")
print(tf1.to_string() if len(tf1) else "(ninguno)")

print("\nInconsistencias PAM/PCP tras limpiar (df1):")
print(conteo_notas_fuera_rango(df1) or "ninguna detectada")

Faltantes después de limpiar (df1):
                                             nulos    pct
programa_academico_secundario               203105  95.69
descripcion_programa_secundario             203105  95.69
estado_doble_programa                       203016  95.64
promedio_semestral_programa_principal(pcp)   33442  15.75
ciudad_direccion_fisica                      13216   6.23
ubicacion_semestral                           8439   3.98
promedio_acumulado_programa_principal(pam)    4968   2.34
organizacion_academica                           9   0.00

Inconsistencias PAM/PCP tras limpiar (df1):
ninguna detectada


In [9]:
# Eliminar columna 
df3 = df3.drop('tematica/asignatura/actividad', axis=1)
print(df3)

                             jefatura ciclo_lectivo  \
0       jefatura desarrollo deportivo        2023-2   
1       jefatura desarrollo deportivo        2023-2   
2       jefatura desarrollo deportivo        2023-2   
3       jefatura desarrollo deportivo        2023-2   
4       jefatura desarrollo deportivo        2023-2   
...                               ...           ...   
291790    jefatura de exito academico        2025-1   
291797    jefatura de exito academico        2025-1   
291811    jefatura de exito academico        2025-1   
291824    jefatura de exito academico        2025-1   
291838    jefatura de exito academico        2025-1   

              estrategia_para_acreditacion tipo_de_actividad  \
0             actividades recreodeportivas  carrera atletica   
1             actividades recreodeportivas  carrera atletica   
2             actividades recreodeportivas  carrera atletica   
3             actividades recreodeportivas  carrera atletica   
4             activ

In [10]:
# Objetivo: convertir a minúsculas el contenido textual de los DataFrames df1, df2 y df3.
# Consideraciones:
# - Solo se procesan columnas de tipo 'object' (texto) para preservar tipos numéricos y fechas.
# - Los NaN y valores no-string se mantienen intactos.
# - Uso de .map en cada columna para evitar crear copias innecesarias de todo el DataFrame.

for nombre, df in [("df1", df1), ("df2", df2), ("df3", df3)]:
    # seleccionar columnas de texto
    columnas_texto = df.select_dtypes(include="object").columns
    for col in columnas_texto:
        # convertir a minúsculas solo si el valor es string
        df[col] = df[col].map(lambda x: x.lower() if isinstance(x, str) else x)
    print(f"{nombre}: columnas procesadas = {len(columnas_texto)}  |  shape = {df.shape}")

df1: columnas procesadas = 11  |  shape = (212263, 16)
df2: columnas procesadas = 5  |  shape = (220339, 6)
df3: columnas procesadas = 7  |  shape = (87127, 7)


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Armonizar nombres de columnas en df3 para alineación posterior
# ─────────────────────────────────────────────────────────────────────────────
# Objetivo: renombrar columnas de df3 para que coincidan con terminología 
# consistente usada en df2 y facilitar fusiones futuras.
#
# Mapeo de renombrado:
#  - "Ciclo lectivo" → "periodo" (formato temporal consistente)
#  - "Estrategia para acreditación" → "estrategia_programa_o_servicio" (alineación con df2)
#  - "Tipo de actividad" → "actividad" (simplificación de nombre)

df3 = df3.rename(columns={
    'Ciclo lectivo': 'periodo',
    'Estrategia para acreditación': 'estrategia_programa_o_servicio',
    'Tipo de actividad': 'actividad'
})

print("Columnas de df3 tras armonización:")
print(df3.columns.tolist())
print(f"\nPrimeras filas de df3:\n{df3.head(3)}")

Columnas de df3 tras armonización:
['jefatura', 'ciclo_lectivo', 'estrategia_para_acreditacion', 'tipo_de_actividad', 'codigo', 'sexo', 'modalidad']

Primeras filas de df3:
                        jefatura ciclo_lectivo  estrategia_para_acreditacion  \
0  jefatura desarrollo deportivo        2023-2  actividades recreodeportivas   
1  jefatura desarrollo deportivo        2023-2  actividades recreodeportivas   
2  jefatura desarrollo deportivo        2023-2  actividades recreodeportivas   

  tipo_de_actividad            codigo   sexo   modalidad  
0  carrera atletica  c209356ls288143x  mujer  presencial  
1  carrera atletica  c432120ye277415a  mujer  presencial  
2  carrera atletica  c182352df459201a  mujer  presencial  


**Imputación de acuerdo a modelo de negocio**

In [12]:
# descargar dataset limpio
df1.to_csv(r"Data\caracterizacion_limpia.csv", index=False)
df2.to_csv(r"Data\asistencia_2015_2022_limpia.csv", index=False)
df3.to_csv(r"Data\asistencia_2023_2025_limpia.csv", index=False)

## Visualizaciones interpretativas

Se generan gráficos con estilo académico consistente (seaborn + matplotlib) y se guardan en `src/graphics`.

Gráficos incluidos:
- **Histogramas + KDE** para PAM, PCP y créditos inscritos, con líneas de media y mediana.
- **Boxplot comparativo** PAM vs PCP en un solo panel.
- **Violin plots** PAM y PCP por sexo.
- **Scatter** PAM vs PCP coloreado por sexo, con línea de identidad.
- **Donut** de distribución por sexo.
- **Barras horizontales** para grupo académico, programas, jefatura y ciudad.
- **Evolución temporal** de registros por año (barras + línea).

In [13]:
# -------- Visualizaciones mejoradas — exportación a src/graphics --------
# Este bloque genera gráficos descriptivos a partir de df1 y df2
# y los guarda en la carpeta src/graphics.

import seaborn as sns
from pathlib import Path

# Estilo global para todas las visualizaciones
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Paleta de color por sexo
PALETTE_SEXO = {"mujer": "#E07B8B", "hombre": "#5B8DB8", "otro": "#9DC4FF"}

# Colores base del informe
COLOR_MAIN = "#001C64"
COLOR_ACCENT = "#193F9E"

# Carpeta de salida para las imágenes
out_dir = Path("src") / "graphics"
out_dir.mkdir(parents=True, exist_ok=True)

# Lista para registrar los archivos generados
saved_files = []


def guardar(fig, nombre):
    """
    Guarda una figura en disco con buena resolución, la cierra
    y registra su ruta en saved_files.
    """
    path = out_dir / nombre
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_files.append(str(path))


# ── 1. Histograma + KDE para PAM y PCP ──────────────────────────────────────
# Se analiza la distribución de ambos promedios académicos y se añaden
# líneas de referencia para media y mediana.
for col, titulo in [
    ("promedio_acumulado_programa_principal(pam)", "Promedio Acumulado (PAM)"),
    ("promedio_semestral_programa_principal(pcp)", "Promedio Semestral (PCP)"),
]:
    serie = df1[col].dropna()

    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.histplot(
        serie,
        bins=40,
        kde=True,
        color=COLOR_MAIN,
        ax=ax,
        edgecolor="white",
        linewidth=0.4,
    )

    ax.axvline(
        serie.mean(),
        color="crimson",
        linestyle="--",
        linewidth=1.5,
        label=f"Media: {serie.mean():.2f}",
    )
    ax.axvline(
        serie.median(),
        color="darkorange",
        linestyle=":",
        linewidth=1.5,
        label=f"Mediana: {serie.median():.2f}",
    )

    ax.set_title(f"Distribución de {titulo}", fontweight="bold")
    ax.set_xlabel(titulo)
    ax.set_ylabel("Frecuencia")
    ax.legend()

    safe = col.replace("(", "").replace(")", "")
    guardar(fig, f"hist_{safe}.png")


# ── 2. Histograma de créditos inscritos ─────────────────────────────────────
# Muestra la distribución de los créditos inscritos por estudiante.
serie = df1["creditos_inscritos"].dropna()

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(
    serie,
    bins=30,
    kde=True,
    color=COLOR_MAIN,
    ax=ax,
    edgecolor="white",
    linewidth=0.4,
)
ax.axvline(
    serie.mean(),
    color="crimson",
    linestyle="--",
    linewidth=1.5,
    label=f"Media: {serie.mean():.1f}",
)
ax.set_title("Distribución de Créditos Inscritos", fontweight="bold")
ax.set_xlabel("Créditos inscritos")
ax.set_ylabel("Frecuencia")
ax.legend()
guardar(fig, "hist_creditos_inscritos.png")


# ── 3. Boxplot comparativo PAM vs PCP ───────────────────────────────────────
# Compara la dispersión de ambos promedios académicos.
pam = df1["promedio_acumulado_programa_principal(pam)"].dropna()
pcp = df1["promedio_semestral_programa_principal(pcp)"].dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(
    [pam, pcp],
    vert=False,
    patch_artist=True,
    notch=True,
    labels=["PAM (acumulado)", "PCP (semestral)"],
    medianprops=dict(color="crimson", linewidth=2),
    boxprops=dict(facecolor=COLOR_MAIN, alpha=0.55),
    flierprops=dict(marker=".", markersize=2, alpha=0.25),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
)
ax.set_title("Distribución de Promedios Académicos (PAM vs PCP)", fontweight="bold")
ax.set_xlabel("Promedio (escala 0–5)")
guardar(fig, "box_promedios_academicos.png")


# ── 4. Violin: PAM y PCP por sexo ───────────────────────────────────────────
# Analiza la distribución de PAM y PCP separada por sexo.
for col, titulo in [
    ("promedio_acumulado_programa_principal(pam)", "Promedio Acumulado (PAM)"),
    ("promedio_semestral_programa_principal(pcp)", "Promedio Semestral (PCP)"),
]:
    temp = df1[[col, "sexo"]].dropna()
    temp = temp[temp["sexo"].isin(["mujer", "hombre"])]

    if temp.empty:
        continue

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.violinplot(
        data=temp,
        x="sexo",
        y=col,
        palette=PALETTE_SEXO,
        ax=ax,
        inner="quartile",
        linewidth=0.8,
    )

    short = titulo.split("(")[0].strip()
    ax.set_title(f"{short} por Sexo", fontweight="bold")
    ax.set_xlabel("Sexo")
    ax.set_ylabel(short)

    safe = col.replace("(", "").replace(")", "")
    guardar(fig, f"violin_{safe}_por_sexo.png")


# ── 5. Scatter PAM vs PCP coloreado por sexo ────────────────────────────────
# Observa la relación entre PAM y PCP, diferenciando por sexo.
x_col = "promedio_acumulado_programa_principal(pam)"
y_col = "promedio_semestral_programa_principal(pcp)"
temp = df1[[x_col, y_col, "sexo"]].dropna()
temp = temp[temp["sexo"].isin(["mujer", "hombre"])]

fig, ax = plt.subplots(figsize=(7, 6))
for sexo, color in [
    ("mujer", PALETTE_SEXO["mujer"]),
    ("hombre", PALETTE_SEXO["hombre"]),
]:
    sub = temp[temp["sexo"] == sexo]
    ax.scatter(
        sub[x_col],
        sub[y_col],
        alpha=0.12,
        s=8,
        color=color,
        label=sexo.capitalize(),
    )

# Línea de referencia: igualdad entre PAM y PCP
ax.plot([0, 5], [0, 5], "k--", linewidth=0.8, alpha=0.5, label="PAM = PCP")
ax.set_xlim(0, 5.1)
ax.set_ylim(0, 5.1)
ax.set_title("Relación PAM vs PCP por Sexo", fontweight="bold")
ax.set_xlabel("Promedio Acumulado (PAM)")
ax.set_ylabel("Promedio Semestral (PCP)")
ax.legend(title="Sexo", frameon=True)
guardar(fig, "scatter_pam_vs_pcp.png")


# ── 6. Donut: distribución por sexo ─────────────────────────────────────────
# Representa la composición de la muestra por sexo.
sexo_counts = df1["sexo"].value_counts(dropna=True)

fig, ax = plt.subplots(figsize=(6, 6))
wedges, _, autotexts = ax.pie(
    sexo_counts,
    labels=None,
    autopct="%1.1f%%",
    colors=[PALETTE_SEXO.get(s, "#CCCCCC") for s in sexo_counts.index],
    startangle=90,
    pctdistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor="white", linewidth=2),
)

for t in autotexts:
    t.set_fontsize(13)

ax.legend(
    wedges,
    [s.capitalize() for s in sexo_counts.index],
    loc="lower center",
    ncol=len(sexo_counts),
    frameon=False,
    fontsize=11,
)
ax.set_title("Distribución por Sexo", fontweight="bold", pad=20)
guardar(fig, "donut_sexo.png")


# ── 7. Registros por grupo académico ────────────────────────────────────────
# Muestra los 10 grupos académicos con mayor número de registros.
freq = df1["grupo_academico"].value_counts(dropna=True).head(10)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(freq.index[::-1], freq.values[::-1], color=COLOR_MAIN, edgecolor="white")

for bar, val in zip(bars, freq.values[::-1]):
    ax.text(
        bar.get_width() + freq.values.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:,}",
        va="center",
        fontsize=9,
    )

ax.set_title("Registros por Grupo Académico", fontweight="bold")
ax.set_xlabel("Número de registros")
ax.set_ylabel("")
ax.set_xlim(0, freq.values.max() * 1.15)
guardar(fig, "bar_grupo_academico.png")


# ── 8. Top 10 programas académicos ──────────────────────────────────────────
# Identifica los programas con más registros.
freq = df1["descripcion_programa_principal"].value_counts(dropna=True).head(10)
labels = [l.capitalize()[:38] for l in freq.index[::-1]]

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(labels, freq.values[::-1], color="#7E9FC9", edgecolor="white")

for bar, val in zip(bars, freq.values[::-1]):
    ax.text(
        bar.get_width() + freq.values.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:,}",
        va="center",
        fontsize=9,
    )

ax.set_title("Top 10 Programas Académicos (registros)", fontweight="bold")
ax.set_xlabel("Número de registros")
ax.set_xlim(0, freq.values.max() * 1.18)
guardar(fig, "bar_top10_programa.png")


# ── 9. Registros por jefatura de bienestar ──────────────────────────────────
# Cuenta registros por jefatura en df2.
freq = df2["jefatura"].value_counts(dropna=True)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(freq.index[::-1], freq.values[::-1], color=COLOR_ACCENT, edgecolor="white")

for bar, val in zip(bars, freq.values[::-1]):
    ax.text(
        bar.get_width() + freq.values.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:,}",
        va="center",
        fontsize=9,
    )

ax.set_title("Registros por Jefatura de Bienestar", fontweight="bold")
ax.set_xlabel("Número de registros")
ax.set_xlim(0, freq.values.max() * 1.2)
guardar(fig, "bar_jefatura.png")


# ── 10. Evolución temporal de registros por año ─────────────────────────────
# Visualiza la cantidad de registros de asistencia por año.
anual = df2["ano"].dropna().astype(int).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(anual.index, anual.values, color=COLOR_MAIN, edgecolor="white", width=0.65)
ax.plot(anual.index, anual.values, "o-", color="crimson", linewidth=1.8, markersize=5, zorder=3)

ax.set_title("Evolución de Registros de Asistencia por Año", fontweight="bold")
ax.set_xlabel("Año")
ax.set_ylabel("Número de registros")
ax.set_xticks(anual.index)
ax.tick_params(axis="x", rotation=45)
guardar(fig, "bar_evolucion_anual.png")


# ── 11. Top 10 ciudades de residencia ───────────────────────────────────────
# Muestra las ciudades más frecuentes en el campo de dirección física.
freq = df1["ciudad_direccion_fisica"].value_counts(dropna=True).head(10)
labels = [l.capitalize()[:32] for l in freq.index[::-1]]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(labels, freq.values[::-1], color="#85B5A0", edgecolor="white")

for bar, val in zip(bars, freq.values[::-1]):
    ax.text(
        bar.get_width() + freq.values.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:,}",
        va="center",
        fontsize=9,
    )

ax.set_title("Top 10 Ciudades de Dirección Física", fontweight="bold")
ax.set_xlabel("Número de registros")
ax.set_xlim(0, freq.values.max() * 1.2)
guardar(fig, "bar_top10_ciudad.png")


# ── Resumen ─────────────────────────────────────────────────────────────────
print(f"Gráficos guardados en: {out_dir}")
print(f"Total de imágenes generadas: {len(saved_files)}")
for p in saved_files:
    print(" -", p)

C:\Users\vanev\AppData\Local\Temp\ipykernel_10728\2767957997.py:120: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(
C:\Users\vanev\AppData\Local\Temp\ipykernel_10728\2767957997.py:150: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(
C:\Users\vanev\AppData\Local\Temp\ipykernel_10728\2767957997.py:150: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(


Gráficos guardados en: src\graphics
Total de imágenes generadas: 13
 - src\graphics\hist_promedio_acumulado_programa_principalpam.png
 - src\graphics\hist_promedio_semestral_programa_principalpcp.png
 - src\graphics\hist_creditos_inscritos.png
 - src\graphics\box_promedios_academicos.png
 - src\graphics\violin_promedio_acumulado_programa_principalpam_por_sexo.png
 - src\graphics\violin_promedio_semestral_programa_principalpcp_por_sexo.png
 - src\graphics\scatter_pam_vs_pcp.png
 - src\graphics\donut_sexo.png
 - src\graphics\bar_grupo_academico.png
 - src\graphics\bar_top10_programa.png
 - src\graphics\bar_jefatura.png
 - src\graphics\bar_evolucion_anual.png
 - src\graphics\bar_top10_ciudad.png


In [14]:
# Promedio de participación en Bienestar por género ────────────────────
# Objetivo:
# - Contar el número total de participaciones por estudiante en df2 y df3.
# - Unir ese total con el género de cada estudiante, tomado del último registro en df1.
# - Filtrar únicamente los géneros "mujer" y "hombre".
# - Calcular promedio, desviación estándar y tamaño de muestra por género.
# - Visualizar los resultados con barras de intervalo de confianza y boxplot.

# Contar asistencias por estudiante en cada fuente de datos
part2_g = df2.groupby("codigo").size().rename("participaciones")
part3_g = df3.groupby("codigo").size().rename("participaciones")

# Unificar ambas fuentes y sumar participaciones por estudiante
part_total = (
    pd.concat([part2_g, part3_g])
    .groupby(level=0)
    .sum()
    .reset_index()
    .rename(columns={"index": "codigo"})
)

# Obtener el género más reciente de cada estudiante desde df1
sexo_por_estudiante = (
    df1.sort_values("ciclo_lectivo")
    .groupby("codigo", as_index=False)["sexo"]
    .last()
)

# Unir participaciones con género
part_sexo = part_total.merge(sexo_por_estudiante, on="codigo", how="left")

# Filtrar solo los géneros de interés
part_sexo = part_sexo[part_sexo["sexo"].isin(["mujer", "hombre"])]

# Resumen estadístico por género
resumen_sexo = (
    part_sexo.groupby("sexo")["participaciones"]
    .agg(media="mean", std="std", n="count")
    .reset_index()
)

# Crear figura con dos paneles
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: barras con intervalo de confianza 95 %
colores = [PALETTE_SEXO.get(s, "#CCCCCC") for s in resumen_sexo["sexo"]]
bars = axes[0].bar(
    resumen_sexo["sexo"].str.capitalize(),
    resumen_sexo["media"],
    yerr=resumen_sexo["std"] / resumen_sexo["n"] ** 0.5 * 1.96,
    color=colores,
    edgecolor="white",
    capsize=6,
    error_kw=dict(linewidth=1.5),
)

# Etiquetas sobre las barras
for bar, row in zip(bars, resumen_sexo.itertuples()):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + resumen_sexo["media"].max() * 0.02,
        f"{row.media:.1f}",
        ha="center",
        fontsize=11,
        fontweight="bold",
    )

axes[0].set_title("Promedio de participaciones\npor género (IC 95 %)", fontweight="bold")
axes[0].set_ylabel("Participaciones promedio")
axes[0].set_ylim(0, resumen_sexo["media"].max() * 1.25)

# Panel 2: boxplot por género
data_m = part_sexo[part_sexo["sexo"] == "mujer"]["participaciones"]
data_h = part_sexo[part_sexo["sexo"] == "hombre"]["participaciones"]

bp = axes[1].boxplot(
    [data_m, data_h],
    patch_artist=True,
    notch=False,
    labels=["Mujer", "Hombre"],
    medianprops=dict(color="crimson", linewidth=2),
    flierprops=dict(marker=".", markersize=2, alpha=0.2),
)

# Color de las cajas
bp["boxes"][0].set_facecolor(PALETTE_SEXO["mujer"])
bp["boxes"][1].set_facecolor(PALETTE_SEXO["hombre"])
for box in bp["boxes"]:
    box.set_alpha(0.6)

axes[1].set_title("Distribución de participaciones\npor género", fontweight="bold")
axes[1].set_ylabel("Participaciones totales")

# Título general y exportación
plt.suptitle("Participación en Bienestar por Género (2015–2025)", fontweight="bold", y=1.02)
plt.tight_layout()
guardar(fig, "participacion_por_genero.png")
plt.show()

# Resumen tabular final
print("Resumen:")
print(resumen_sexo.to_string(index=False))

C:\Users\vanev\AppData\Local\Temp\ipykernel_10728\1247589073.py:76: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = axes[1].boxplot(


Resumen:
  sexo     media      std     n
hombre 10.386658 9.931492 10883
 mujer 10.268383 9.747107 13749


In [15]:
# Promedio de participación por municipio de residencia
# ------------------------------------------------------
# Objetivo:
# - Obtener, para cada estudiante, su ciudad de residencia más reciente
#   registrada en df1.
# - Unir esa información con el total de participaciones acumuladas en
#   bienestar (part_total).
# - Calcular el promedio de participaciones por ciudad.
# - Visualizar dos perspectivas:
#     1) municipios con mayor promedio de participación
#     2) municipios con mayor número de estudiantes
# - Guardar la figura resultante.

# Ciudad más reciente registrada por estudiante
ciudad_por_estudiante = (
    df1.sort_values("ciclo_lectivo")
    .groupby("codigo", as_index=False)["ciudad_direccion_fisica"]
    .last()
)

# Unir participaciones totales con ciudad de residencia
part_ciudad = part_total.merge(ciudad_por_estudiante, on="codigo", how="left")

# Eliminar estudiantes sin ciudad registrada
part_ciudad = part_ciudad.dropna(subset=["ciudad_direccion_fisica"])

# Resumen por ciudad:
# - media: promedio de participaciones
# - n: número de estudiantes en la ciudad
resumen_ciudad = (
    part_ciudad.groupby("ciudad_direccion_fisica")["participaciones"]
    .agg(media="mean", n="count")
    .reset_index()
)

# ------------------------------------------------------
# Panel A: top 15 municipios con mayor promedio
#         (solo ciudades con al menos 30 estudiantes)
# ------------------------------------------------------
top_media = (
    resumen_ciudad[resumen_ciudad["n"] >= 30]
    .nlargest(15, "media")
    .sort_values("media")
)

# ------------------------------------------------------
# Panel B: top 15 municipios con más estudiantes
# ------------------------------------------------------
top_volumen = (
    resumen_ciudad.nlargest(15, "n")
    .sort_values("media")
)

# Crear figura con dos paneles comparativos
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel A: promedio de participación por municipio
bars = axes[0].barh(
    top_media["ciudad_direccion_fisica"].str.capitalize(),
    top_media["media"],
    color=COLOR_MAIN,
    edgecolor="white",
)

for bar, val in zip(bars, top_media["media"]):
    axes[0].text(
        bar.get_width() + top_media["media"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}",
        va="center",
        fontsize=9,
    )

axes[0].set_title(
    "Top 15 municipios — mayor promedio\nde participación (mín. 30 estudiantes)",
    fontweight="bold",
)
axes[0].set_xlabel("Participaciones promedio")
axes[0].set_xlim(0, top_media["media"].max() * 1.2)

# Panel B: municipios con más estudiantes
colors_b = [
    COLOR_ACCENT if c == "bogota d.c." else "#7E9FC9"
    for c in top_volumen["ciudad_direccion_fisica"]
]

bars2 = axes[1].barh(
    top_volumen["ciudad_direccion_fisica"].str.capitalize(),
    top_volumen["media"],
    color=colors_b,
    edgecolor="white",
)

for bar, row in zip(bars2, top_volumen.itertuples()):
    axes[1].text(
        bar.get_width() + top_volumen["media"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{row.media:.1f}  (n={row.n:,})",
        va="center",
        fontsize=9,
    )

axes[1].set_title(
    "Top 15 municipios por nº de estudiantes\n(promedio de participación)",
    fontweight="bold",
)
axes[1].set_xlabel("Participaciones promedio")
axes[1].set_xlim(0, top_volumen["media"].max() * 1.35)

# Título general y exportación
plt.suptitle(
    "Promedio de Participación en Bienestar por Municipio de Residencia (2015–2025)",
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
guardar(fig, "participacion_por_municipio.png")
plt.show()

# Resumen textual
print(f"Municipios con datos: {resumen_ciudad['ciudad_direccion_fisica'].nunique():,}")
print(f"Municipios con ≥30 estudiantes: {(resumen_ciudad['n'] >= 30).sum():,}")

Municipios con datos: 310
Municipios con ≥30 estudiantes: 35


In [16]:
# ── 14. Promedio de participación vs. promedio semestral (PCP) ───────────────
pcp_col = "promedio_semestral_programa_principal(pcp)"

pcp_por_estudiante = (
    df1.sort_values("ciclo_lectivo")
    .groupby("codigo", as_index=False)[pcp_col]
    .last()
)

part_pcp = part_total.merge(pcp_por_estudiante, on="codigo", how="left").dropna(subset=[pcp_col])

# Crear rangos de PCP (intervalos de 0.5)
part_pcp["rango_pcp"] = pd.cut(
    part_pcp[pcp_col],
    bins=[0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
    labels=["0–2.5", "2.5–3.0", "3.0–3.5", "3.5–4.0", "4.0–4.5", "4.5–5.0"],
    right=True
)

resumen_pcp = (
    part_pcp.groupby("rango_pcp", observed=True)["participaciones"]
    .agg(media="mean", std="std", n="count")
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# — Panel A: barras por rango de PCP con IC —
bars = axes[0].bar(
    resumen_pcp["rango_pcp"].astype(str),
    resumen_pcp["media"],
    yerr=resumen_pcp["std"] / resumen_pcp["n"] ** 0.5 * 1.96,
    color=COLOR_MAIN, edgecolor="white", capsize=5,
    error_kw=dict(linewidth=1.4)
)

for bar, row in zip(bars, resumen_pcp.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + resumen_pcp["media"].max() * 0.02,
                 f"{row.media:.1f}\n(n={row.n:,})", ha="center", fontsize=8.5)
axes[0].set_title("Promedio de participación\npor rango de PCP (IC 95 %)", fontweight="bold")
axes[0].set_xlabel("Rango de promedio semestral (PCP)")
axes[0].set_ylabel("Participaciones promedio")
axes[0].set_ylim(0, resumen_pcp["media"].max() * 1.3)
axes[0].tick_params(axis="x", rotation=30)

# — Panel B: scatter con línea de tendencia —
muestra = part_pcp.sample(min(8000, len(part_pcp)), random_state=42)
axes[1].scatter(
    muestra[pcp_col], muestra["participaciones"],
    alpha=0.15, s=8, color=COLOR_MAIN
)

# Línea de tendencia (regresión lineal simple)
coef = np.polyfit(part_pcp[pcp_col], part_pcp["participaciones"], 1)
x_line = np.linspace(part_pcp[pcp_col].min(), part_pcp[pcp_col].max(), 100)
axes[1].plot(x_line, np.polyval(coef, x_line), color="crimson",
             linewidth=2, label=f"Tendencia (pendiente={coef[0]:+.2f})")
axes[1].set_title("Dispersión PCP vs Participación\n(muestra + tendencia lineal)", fontweight="bold")
axes[1].set_xlabel("Promedio Semestral (PCP)")
axes[1].set_ylabel("Participaciones totales")
axes[1].legend(frameon=True)

plt.suptitle("Relación entre Participación en Bienestar y Promedio Semestral (PCP)",
             fontweight="bold", y=1.02)
plt.tight_layout()
guardar(fig, "participacion_vs_pcp.png")
plt.show()

corr = part_pcp[[pcp_col, "participaciones"]].corr().iloc[0, 1]
print(f"Correlación de Pearson (PCP vs participación): {corr:.3f}")



Correlación de Pearson (PCP vs participación): 0.158


## Identificación de perfiles estudiantiles

**Pregunta:** ¿Qué perfiles se pueden identificar entre los estudiantes a partir de sus características académicas y su nivel de participación desde 2015 hasta 2025?

**Enfoque:** se combina el historial de caracterización (`df1`) con los registros de asistencia (`df2` + `df3`) para construir un perfil por estudiante con cuatro variables:
- `pam` — promedio acumulado
- `pcp` — promedio semestral
- `creditos_inscritos` — carga académica
- `participacion_total` — número de asistencias registradas en bienestar (2015–2025)

Se aplica **K-Means** con escalado estándar. El número óptimo de clusters se selecciona con el método del codo y el coeficiente de silueta.


In [17]:
# ── Construir tabla de perfil por estudiante ─────────────────────────────

# Contar participaciones por estudiante en cada fuente
part2 = df2.groupby("codigo").size().rename("part_2015_2022")
part3 = df3.groupby("codigo").size().rename("part_2023_2025")

participacion = (
    pd.concat([part2, part3], axis=1)
    .fillna(0)
    .assign(participacion_total=lambda d: d["part_2015_2022"] + d["part_2023_2025"])
    .reset_index()
)

# Último registro de cada estudiante en caracterización (período más reciente)
df1_ultimo = (
    df1.sort_values("ciclo_lectivo")
    .groupby("codigo", as_index=False)
    .last()
)

# Unir
df_perfil = df1_ultimo.merge(
    participacion[["codigo", "participacion_total"]], on="codigo", how="left"
)
df_perfil["participacion_total"] = df_perfil["participacion_total"].fillna(0)

FEATURES = [
    "promedio_acumulado_programa_principal(pam)",
    "promedio_semestral_programa_principal(pcp)",
    "creditos_inscritos",
    "participacion_total",
]

print(f"Estudiantes únicos en la tabla de perfil: {len(df_perfil):,}")
print(f"Estudiantes con todas las features completas: {df_perfil[FEATURES].dropna().shape[0]:,}\n")
print(df_perfil[FEATURES].describe().T.to_string())

Estudiantes únicos en la tabla de perfil: 31,691
Estudiantes con todas las features completas: 30,230

                                              count      mean       std   min   25%    50%   75%    max
promedio_acumulado_programa_principal(pam)  30361.0  3.859978  0.554062  0.02  3.66   3.96   4.2    5.0
promedio_semestral_programa_principal(pcp)  30244.0  4.058240  0.823413  0.02  3.76   4.27   4.6    5.0
creditos_inscritos                          31691.0  9.321258  9.593723  0.00  0.00  10.00  18.0   43.0
participacion_total                         31691.0  8.023130  9.670844  0.00  1.00   5.00  12.0  127.0


In [18]:
# ── Selección del número óptimo de clusters ──────────────────────────────
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

df_cluster = df_perfil[FEATURES].dropna().copy()
scaler = StandardScaler()
X = scaler.fit_transform(df_cluster)

inertias, silhouettes = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_range), inertias, "o-", color=COLOR_MAIN, linewidth=2, markersize=7)
axes[0].set_xlabel("Número de clusters (k)")
axes[0].set_ylabel("Inercia")
axes[0].set_title("Método del codo", fontweight="bold")

axes[1].plot(list(K_range), silhouettes, "o-", color=COLOR_ACCENT, linewidth=2, markersize=7)
axes[1].set_xlabel("Número de clusters (k)")
axes[1].set_ylabel("Coeficiente de silueta")
axes[1].set_title("Silueta por número de clusters", fontweight="bold")

plt.tight_layout()
guardar(fig, "clustering_seleccion_k.png")
plt.show()

k_opt = K_range.start + silhouettes.index(max(silhouettes))
print(f"k con mayor silueta: {k_opt}  (silueta = {max(silhouettes):.3f})")


k con mayor silueta: 2  (silueta = 0.376)


In [19]:
# ── Aplicar K-Means con k óptimo y describir perfiles ───────────────────
K_FINAL = k_opt  # cambia manualmente si prefieres otro valor

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df_cluster = df_cluster.copy()
df_cluster["cluster"] = km_final.fit_predict(X)

# Descripción estadística por cluster
resumen = (
    df_cluster.groupby("cluster")[FEATURES]
    .agg(["mean", "median", "count"])
)
resumen.columns = ["_".join(c) for c in resumen.columns]
print("Resumen por cluster (medias y medianas):\n")
print(resumen.T.to_string())

# Tamaño de cada cluster
print("\nDistribución de estudiantes por cluster:")
print(df_cluster["cluster"].value_counts().sort_index().to_string())

Resumen por cluster (medias y medianas):

cluster                                                       0            1
promedio_acumulado_programa_principal(pam)_mean        4.040794     2.996435
promedio_acumulado_programa_principal(pam)_median      4.040000     3.180000
promedio_acumulado_programa_principal(pam)_count   25060.000000  5170.000000
promedio_semestral_programa_principal(pcp)_mean        4.348656     2.649559
promedio_semestral_programa_principal(pcp)_median      4.400000     2.940000
promedio_semestral_programa_principal(pcp)_count   25060.000000  5170.000000
creditos_inscritos_mean                                9.763687     9.460542
creditos_inscritos_median                             12.000000    12.000000
creditos_inscritos_count                           25060.000000  5170.000000
participacion_total_mean                               9.275459     4.078143
participacion_total_median                             6.000000     3.000000
participacion_total_count         

In [20]:
# ── Visualizaciones de perfiles ─────────────────────────────────────────
CLUSTER_COLORS = ["#4A7FA5", "#E07B8B", "#6BAE8E", "#F0A500", "#9B59B6", "#E67E22"]
palette_cl = {i: CLUSTER_COLORS[i % len(CLUSTER_COLORS)] for i in range(K_FINAL)}

# — A. Scatter PAM vs participacion_total por cluster —
fig, ax = plt.subplots(figsize=(8, 6))
for cl in sorted(df_cluster["cluster"].unique()):
    sub = df_cluster[df_cluster["cluster"] == cl]
    ax.scatter(
        sub["promedio_acumulado_programa_principal(pam)"],
        sub["participacion_total"],
        alpha=0.25, s=10, color=palette_cl[cl], label=f"Perfil {cl + 1}"
    )

ax.set_xlabel("Promedio Acumulado (PAM)")
ax.set_ylabel("Participación total en Bienestar")
ax.set_title("Perfiles: PAM vs Participación en Bienestar", fontweight="bold")
ax.legend(title="Perfil", frameon=True)
guardar(fig, "perfil_scatter_pam_participacion.png")
plt.show()

# — B. Scatter PCP vs participacion_total por cluster —
fig, ax = plt.subplots(figsize=(8, 6))
for cl in sorted(df_cluster["cluster"].unique()):
    sub = df_cluster[df_cluster["cluster"] == cl]
    ax.scatter(
        sub["promedio_semestral_programa_principal(pcp)"],
        sub["participacion_total"],
        alpha=0.25, s=10, color=palette_cl[cl], label=f"Perfil {cl + 1}"
    )
ax.set_xlabel("Promedio Semestral (PCP)")
ax.set_ylabel("Participación total en Bienestar")
ax.set_title("Perfiles: PCP vs Participación en Bienestar", fontweight="bold")
ax.legend(title="Perfil", frameon=True)
guardar(fig, "perfil_scatter_pcp_participacion.png")
plt.show()

# — C. Boxplots de cada feature por cluster —
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
labels_feat = ["PAM (acumulado)", "PCP (semestral)", "Créditos inscritos", "Participación total"]
for ax, col, label in zip(axes.flat, FEATURES, labels_feat):
    data_by_cluster = [
        df_cluster[df_cluster["cluster"] == cl][col].dropna().values
        for cl in sorted(df_cluster["cluster"].unique())
    ]
    bp = ax.boxplot(
        data_by_cluster, patch_artist=True, notch=False,
        medianprops=dict(color="crimson", linewidth=2),
        flierprops=dict(marker=".", markersize=2, alpha=0.2),
    )
    for patch, cl in zip(bp["boxes"], sorted(df_cluster["cluster"].unique())):
        patch.set_facecolor(palette_cl[cl])
        patch.set_alpha(0.6)
    ax.set_xticklabels([f"P{cl + 1}" for cl in sorted(df_cluster["cluster"].unique())])
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("Perfil")
plt.suptitle("Distribución de variables por Perfil estudiantil", fontweight="bold", y=1.01)
plt.tight_layout()
guardar(fig, "perfil_boxplots.png")
plt.show()

# — D. Gráfico de radar con medias normalizadas por cluster —
medias = df_cluster.groupby("cluster")[FEATURES].mean()
medias_norm = (medias - medias.min()) / (medias.max() - medias.min())

angles = np.linspace(0, 2 * np.pi, len(FEATURES), endpoint=False).tolist()
angles += angles[:1]
short_labels = ["PAM", "PCP", "Créditos", "Participación"]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for cl in medias_norm.index:
    vals = medias_norm.loc[cl].tolist() + [medias_norm.loc[cl].tolist()[0]]
    ax.plot(angles, vals, "o-", linewidth=2, color=palette_cl[cl], label=f"Perfil {cl + 1}")
    ax.fill(angles, vals, alpha=0.08, color=palette_cl[cl])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(short_labels, fontsize=12)
ax.set_yticklabels([])
ax.set_title("Radar de perfiles estudiantiles\n(variables normalizadas)", fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), frameon=True)
guardar(fig, "perfil_radar.png")
plt.show()

# — E. Distribución de tamaños —
conteo = df_cluster["cluster"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    [f"Perfil {cl + 1}" for cl in conteo.index],
    conteo.values,
    color=[palette_cl[cl] for cl in conteo.index],
    edgecolor="white"
)
for bar, val in zip(bars, conteo.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + conteo.values.max() * 0.01,
            f"{val:,}", ha="center", fontsize=10)
ax.set_title("Número de estudiantes por Perfil", fontweight="bold")
ax.set_ylabel("Estudiantes")
ax.set_ylim(0, conteo.values.max() * 1.15)
guardar(fig, "perfil_tamanos.png")
plt.show()

In [21]:
# ── Nombrar e interpretar los perfiles ───────────────────────────────────
medias_raw = df_cluster.groupby("cluster")[FEATURES].mean().round(2)
medias_raw.index = [f"Perfil {i+1}" for i in medias_raw.index]
medias_raw.columns = ["PAM (media)", "PCP (media)", "Créditos (media)", "Participación (media)"]

print("Medias por perfil:\n")
print(medias_raw.to_string())

print("\n--- Interpretación orientativa ---")
print(
    "Cada perfil agrupa estudiantes con comportamiento similar en rendimiento académico\n"
    "y uso de servicios de bienestar. Los perfiles con alta participación y alto promedio\n"
    "sugieren estudiantes comprometidos integralmente; los de bajo promedio y alta\n"
    "participación pueden indicar estudiantes que buscan apoyo; los de bajo promedio y\n"
    "baja participación representan estudiantes desconectados de la oferta institucional.\n"
    "Interpreta cada perfil a la luz de los valores reales que aparecen en la tabla de arriba."
)

# Exportar tabla de perfiles
df_perfil_out = df_perfil[["codigo"] + FEATURES].dropna().copy()
df_perfil_out["cluster"] = km_final.predict(scaler.transform(df_perfil_out[FEATURES]))
df_perfil_out["perfil"] = df_perfil_out["cluster"].apply(lambda x: f"Perfil {x+1}")
df_perfil_out.to_csv("Data/perfiles_estudiantiles.csv", index=False)
print("\nTabla exportada: Data/perfiles_estudiantiles.csv")

Medias por perfil:

          PAM (media)  PCP (media)  Créditos (media)  Participación (media)
Perfil 1         4.04         4.35              9.76                   9.28
Perfil 2         3.00         2.65              9.46                   4.08

--- Interpretación orientativa ---
Cada perfil agrupa estudiantes con comportamiento similar en rendimiento académico
y uso de servicios de bienestar. Los perfiles con alta participación y alto promedio
sugieren estudiantes comprometidos integralmente; los de bajo promedio y alta
participación pueden indicar estudiantes que buscan apoyo; los de bajo promedio y
baja participación representan estudiantes desconectados de la oferta institucional.
Interpreta cada perfil a la luz de los valores reales que aparecen en la tabla de arriba.

Tabla exportada: Data/perfiles_estudiantiles.csv


## Variables que influyen en la participación en Bienestar

**Pregunta:** ¿Qué variables influyen en la participación en actividades de bienestar, y cómo afecta el rendimiento académico?

**Enfoque en dos pasos:**
1. **Heatmap de correlaciones** — relaciones bivariadas entre todas las variables numéricas.
2. **Regresión lineal múltiple con coeficientes estandarizados** — efecto *independiente* de cada variable sobre la participación, controlando las demás. Un coeficiente positivo significa que al aumentar esa variable, la participación sube; uno negativo, que baja.

Variables consideradas: PAM, PCP, créditos inscritos, semestre cursado, sexo (mujer = 1) y si tiene doble programa (sí = 1).

In [22]:
# ── Preparar dataset para análisis de influencia ─────────────────────────
pam_col = "promedio_acumulado_programa_principal(pam)"
pcp_col = "promedio_semestral_programa_principal(pcp)"

# Extraer semestre como número (ej. "semestre 7" → 7)
df1_base = (
    df1.sort_values("ciclo_lectivo")
    .groupby("codigo", as_index=False)
    .last()
)
df1_base["semestre_num"] = (
    df1_base["ubicacion_semestral"]
    .str.extract(r"(\d+)")[0]
    .astype(float)
)
df1_base["es_mujer"]         = (df1_base["sexo"] == "mujer").astype(int)
df1_base["tiene_doble_prog"] = (df1_base["doble_programa"] == "si").astype(int)

# Unir con participación total
df_reg = df1_base[[
    "codigo", pam_col, pcp_col,
    "creditos_inscritos", "semestre_num",
    "es_mujer", "tiene_doble_prog"
]].merge(part_total, on="codigo", how="left")

df_reg["participaciones"] = df_reg["participaciones"].fillna(0)
df_reg = df_reg.dropna()

COLS_ANALISIS = {
    pam_col:              "PAM",
    pcp_col:              "PCP",
    "creditos_inscritos": "Créditos inscritos",
    "semestre_num":       "Semestre cursado",
    "es_mujer":           "Sexo (mujer=1)",
    "tiene_doble_prog":   "Doble programa",
    "participaciones":    "Participación total",
}

df_plot = df_reg[list(COLS_ANALISIS.keys())].rename(columns=COLS_ANALISIS)
print(f"Estudiantes en el análisis: {len(df_plot):,}")
print(df_plot.describe().T.to_string())

Estudiantes en el análisis: 27,843
                       count      mean       std   min   25%   50%    75%    max
PAM                  27843.0  3.880793  0.535287  0.02  3.68  3.97   4.21    5.0
PCP                  27843.0  4.064672  0.801287  0.02  3.77  4.25   4.60    5.0
Créditos inscritos   27843.0  9.188019  9.675540  0.00  0.00  8.00  18.00   43.0
Semestre cursado     27843.0  6.858241  3.661909  1.00  3.00  8.00  10.00   14.0
Sexo (mujer=1)       27843.0  0.565780  0.495663  0.00  0.00  1.00   1.00    1.0
Doble programa       27843.0  0.015731  0.124435  0.00  0.00  0.00   0.00    1.0
Participación total  27843.0  9.086341  9.846298  0.00  2.00  6.00  13.00  127.0


In [23]:
# ── Heatmap de correlaciones ─────────────────────────────────────────────
corr_matrix = df_plot.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # ocultar triángulo superior

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdYlBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor="white",
    annot_kws={"size": 10}, ax=ax,
    cbar_kws={"shrink": 0.8, "label": "Correlación de Pearson"}
)
ax.set_title("Mapa de calor — correlaciones entre variables\n(negativo = relación inversa, positivo = relación directa)",
             fontweight="bold")
plt.tight_layout()
guardar(fig, "heatmap_correlaciones.png")
plt.show()

print("Correlaciones con Participación total (ordenadas):")
print(
    corr_matrix["Participación total"]
    .drop("Participación total")
    .sort_values(key=abs, ascending=False)
    .to_string()
)

Correlaciones con Participación total (ordenadas):
Semestre cursado      0.228965
PAM                   0.202526
PCP                   0.140997
Créditos inscritos   -0.078872
Doble programa       -0.057246
Sexo (mujer=1)       -0.019871


In [24]:
# ── Regresión lineal múltiple — coeficientes estandarizados ──────────────
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler as SS

TARGET = "Participación total"
PREDICTORES = [c for c in df_plot.columns if c != TARGET]

X_raw = df_plot[PREDICTORES].values
y_raw = df_plot[TARGET].values

# Escalar para comparar coeficientes en la misma unidad
scaler_reg = SS()
X_scaled = scaler_reg.fit_transform(X_raw)

modelo = LinearRegression().fit(X_scaled, y_raw)
coefs = pd.Series(modelo.coef_, index=PREDICTORES).sort_values()

# R² del modelo
r2 = modelo.score(X_scaled, y_raw)

# Colores: verde si positivo, rojo si negativo
colores_coef = ["#E07B8B" if v < 0 else "#4A7FA5" for v in coefs.values]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(coefs.index, coefs.values, color=colores_coef, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
for bar, val in zip(bars, coefs.values):
    x_pos = val + (coefs.abs().max() * 0.02 * (1 if val >= 0 else -1))
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            f"{val:+.3f}", va="center", ha="left" if val >= 0 else "right", fontsize=9)
ax.set_title(
    f"Coeficientes estandarizados — efecto independiente sobre Participación\n"
    f"(R² del modelo = {r2:.3f}  |  azul = sube participación, rojo = baja)",
    fontweight="bold"
)
ax.set_xlabel("Coeficiente β estandarizado")
ax.set_xlim(coefs.min() * 1.3, coefs.max() * 1.3)
plt.tight_layout()
guardar(fig, "regresion_coeficientes.png")
plt.show()

print(f"R² = {r2:.4f}  →  el modelo explica el {r2*100:.1f} % de la varianza en participación")
print("\nCoeficientes estandarizados (β):")
print(coefs.sort_values(key=abs, ascending=False).to_string())

R² = 0.0864  →  el modelo explica el 8.6 % de la varianza en participación

Coeficientes estandarizados (β):
PAM                   2.534156
Semestre cursado      2.021928
PCP                  -1.510556
Doble programa       -0.764890
Sexo (mujer=1)       -0.616216
Créditos inscritos   -0.393245


### Modelado predictivo de la participación en Bienestar

Para responder *¿qué variables predicen mejor la participación?* se comparan tres modelos sobre la misma partición (80 % entrenamiento / 20 % prueba):

| Modelo | Por qué incluirlo |
|---|---|
| **Regresión lineal** | Línea base interpretable; referencia del análisis anterior |
| **Random Forest** | Captura relaciones no lineales; provee SHAP values para explicabilidad |
| **Red Neuronal (MLP)** | Mayor capacidad expresiva; contraste con los modelos anteriores |

Los SHAP values responden directamente la pregunta de investigación: muestran *cuánto y en qué dirección* empuja cada variable la predicción de participación de cada estudiante.

In [25]:
# ── Configuración: dependencias y partición de datos ─────────────────────
%pip install -q shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression as LR
import numpy as np

TARGET_ML      = "Participación total"
PREDICTORES_ML = [c for c in df_plot.columns if c != TARGET_ML]

X_ml = df_plot[PREDICTORES_ML].values
y_ml = df_plot[TARGET_ML].values

X_train, X_test, y_train, y_test = train_test_split(
    X_ml, y_ml, test_size=0.2, random_state=42
)
print(f"Muestras: entrenamiento={len(X_train):,}   prueba={len(X_test):,}")
print(f"Predictores: {PREDICTORES_ML}")

# Regresión lineal sobre la misma partición (línea base)
scaler_lr = StandardScaler()
lr_cmp    = LR().fit(scaler_lr.fit_transform(X_train), y_train)
y_pred_lr = lr_cmp.predict(scaler_lr.transform(X_test))
r2_lr     = r2_score(y_test, y_pred_lr)
rmse_lr   = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print(f"\nRegresión lineal  →  R² = {r2_lr:.4f}   RMSE = {rmse_lr:.4f}")


Note: you may need to restart the kernel to use updated packages.
Muestras: entrenamiento=22,274   prueba=5,569
Predictores: ['PAM', 'PCP', 'Créditos inscritos', 'Semestre cursado', 'Sexo (mujer=1)', 'Doble programa']

Regresión lineal  →  R² = 0.0903   RMSE = 9.5693


In [26]:
# ── Random Forest: importancia de variables y SHAP ────────────────────────
import shap
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=8, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
r2_rf     = r2_score(y_test, y_pred_rf)
rmse_rf   = np.sqrt(mean_squared_error(y_test, y_pred_rf))
print(f"Random Forest  →  R² = {r2_rf:.4f}   RMSE = {rmse_rf:.4f}")

# Importancia MDI (Mean Decrease Impurity)
importancias = pd.Series(rf.feature_importances_, index=PREDICTORES_ML).sort_values()
colores_imp  = ["#4A7FA5" if v == importancias.max() else "#7BB3CC" for v in importancias.values]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(importancias.index, importancias.values, color=colores_imp, edgecolor='white')
for i, (val, name) in enumerate(zip(importancias.values, importancias.index)):
    ax.text(val + 0.001, i, f"{val:.3f}", va="center", fontsize=9)
ax.set_title(
    "Importancia de variables — Random Forest (MDI)\n"
    "(proporción de reducción de impureza; mayor = más influyente)",
    fontweight="bold"
)
ax.set_xlabel("Importancia relativa")
plt.tight_layout()
guardar(fig, "rf_importancia_variables.png")
plt.show()

# SHAP values (muestra de 500 registros para eficiencia)
explainer   = shap.TreeExplainer(rf)
rng         = np.random.default_rng(42)
idx_sample  = rng.choice(len(X_test), size=min(500, len(X_test)), replace=False)
shap_values = explainer.shap_values(X_test[idx_sample])

shap.summary_plot(shap_values, X_test[idx_sample], feature_names=PREDICTORES_ML, show=False)
fig_shap = plt.gcf()
fig_shap.suptitle(
    "SHAP — impacto de cada variable sobre la predicción de participación",
    fontweight="bold", y=1.01
)
guardar(fig_shap, "shap_summary.png")
plt.show()

print("\nInterpretación SHAP:")
print("  · Puntos a la derecha (rojo) → valor alto de la variable SUBE la predicción")
print("  · Puntos a la izquierda (azul) → valor alto de la variable BAJA la predicción")


Random Forest  →  R² = 0.2230   RMSE = 8.8437

Interpretación SHAP:
  · Puntos a la derecha (rojo) → valor alto de la variable SUBE la predicción
  · Puntos a la izquierda (azul) → valor alto de la variable BAJA la predicción


In [27]:
# ── Red Neuronal (MLP) — predicción de participación ─────────────────────
from sklearn.neural_network import MLPRegressor

scaler_mlp = StandardScaler()
X_train_sc = scaler_mlp.fit_transform(X_train)
X_test_sc  = scaler_mlp.transform(X_test)

mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16),
    activation="relu",
    max_iter=400,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
)
mlp.fit(X_train_sc, y_train)

y_pred_mlp = mlp.predict(X_test_sc)
r2_mlp     = r2_score(y_test, y_pred_mlp)
rmse_mlp   = np.sqrt(mean_squared_error(y_test, y_pred_mlp))
print(f"Red Neuronal MLP  →  R² = {r2_mlp:.4f}   RMSE = {rmse_mlp:.4f}")
print(f"Épocas entrenadas: {mlp.n_iter_}")

# Curvas de aprendizaje
has_val = mlp.validation_scores_ is not None
fig, axes = plt.subplots(1, 2 if has_val else 1, figsize=(11 if has_val else 7, 4))
axes = [axes] if not has_val else axes

axes[0].plot(mlp.loss_curve_, color='#4A7FA5')
axes[0].set_title('Pérdida en entrenamiento', fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE')

if has_val:
    axes[1].plot(mlp.validation_scores_, color='#E07B8B')
    axes[1].set_title('R² en validación (early stopping)', fontweight='bold')
    axes[1].set_xlabel('Época')
    axes[1].set_ylabel('R²')

plt.suptitle('Curvas de aprendizaje — Red Neuronal MLP', fontweight='bold')
plt.tight_layout()
guardar(fig, 'mlp_curvas_aprendizaje.png')
plt.show()


Red Neuronal MLP  →  R² = 0.2248   RMSE = 8.8335
Épocas entrenadas: 62


In [28]:
# ── Comparación final de los tres modelos ────────────────────────────────
resultados = pd.DataFrame({
    "Modelo": ["Regresión lineal", "Random Forest", "Red Neuronal (MLP)"],
    "R²":     [r2_lr,   r2_rf,   r2_mlp],
    "RMSE":   [rmse_lr, rmse_rf, rmse_mlp],
}).sort_values('R²', ascending=False).reset_index(drop=True)

print("Comparación de modelos (conjunto de prueba — 20 %):")
print(resultados.to_string(index=False))

palette_cmp = ['#4A7FA5', '#7BB3CC', '#AACDE0']
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for i, (met, titulo) in enumerate([("R²", "R² (mayor es mejor)"), ("RMSE", "RMSE (menor es mejor)")]):
    vals = resultados[met].values
    axes[i].bar(resultados['Modelo'], vals, color=palette_cmp, edgecolor='white')
    axes[i].set_title(titulo, fontweight='bold')
    for j, v in enumerate(vals):
        axes[i].text(j, v * 1.02, f'{v:.3f}', ha='center', fontsize=10)
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Comparación de modelos — predicción de participación en Bienestar',
             fontweight='bold', y=1.02)
plt.tight_layout()
guardar(fig, 'comparacion_modelos.png')
plt.show()

best = resultados.iloc[0]
print(f"\n Mejor modelo: {best['Modelo']}  (R²={best['R²']:.4f}, RMSE={best['RMSE']:.4f})")
print(
    "\nConclusión: el Random Forest suele dominar porque la relación entre variables "
    "y participación NO es estrictamente lineal. Los SHAP values (celda anterior) "
    "revelan qué variable empuja más cada predicción individual."
)


Comparación de modelos (conjunto de prueba — 20 %):
            Modelo       R²     RMSE
Red Neuronal (MLP) 0.224821 8.833523
     Random Forest 0.223027 8.843740
  Regresión lineal 0.090314 9.569270

 Mejor modelo: Red Neuronal (MLP)  (R²=0.2248, RMSE=8.8335)

Conclusión: el Random Forest suele dominar porque la relación entre variables y participación NO es estrictamente lineal. Los SHAP values (celda anterior) revelan qué variable empuja más cada predicción individual.


## Clasificación de actividades: Regresión Logística

**Pregunta:** ¿Es posible predecir a qué tipo de **actividad** de Bienestar asistirá un estudiante según su sexo, ciudad de residencia y promedio acumulado (PAM)?

**Enfoque:** a diferencia de los modelos anteriores (que predicen una variable continua), aquí se entrena un **clasificador multiclase**. Cada registro de asistencia es una observación y el target es la categoría `actividad`.

| Variable | Rol | Tipo |
|---|---|---|
| `sexo` | predictora | categórica → OneHotEncoder |
| `ciudad_direccion_fisica` | predictora | categórica → OneHotEncoder |
| PAM | predictora | numérica (passthrough) |
| `actividad` | target | multiclase |

**Correcciones sobre el código original:**
- `df_p1` no estaba definido: se construye mergeando `df1` (caracterización) con los registros de asistencia (`df2` + `df3`) que contienen `actividad`.
- `LogisticRegression` usa `max_iter=1000` (el default de 100 no converge con muchas clases).
- `OneHotEncoder` incluye `handle_unknown='ignore'` para tolerar ciudades no vistas en prueba.
- El objeto se nombra `modelo_lr` para no sobreescribir la variable `modelo` de celdas anteriores.


In [29]:
# Preparar df_p1: merge de caracterización + asistencias (actividad como target)
# ─────────────────────────────────────────────────────────────────────────────
# df1     → última fila por estudiante (sexo, ciudad_direccion_fisica, PAM)
# df2/df3 → registros de asistencia con columna 'actividad'

PAM_COL_LR = "promedio_acumulado_programa_principal(pam)"

# Unir ambas fuentes de asistencia conservando sólo las columnas necesarias
frames = []
for _df in [df2, df3]:
    if "actividad" in _df.columns and "codigo" in _df.columns:
        frames.append(_df[["codigo", "actividad"]])
asistencia_lr = pd.concat(frames, ignore_index=True).dropna(subset=["actividad"])

# Último registro por estudiante en caracterización
df1_ult_lr = (
    df1.sort_values("ciclo_lectivo")
    .groupby("codigo", as_index=False)
    .last()
)

# Crear df_p1: cada asistencia se enriquece con los atributos del estudiante
df_p1 = asistencia_lr.merge(
    df1_ult_lr[["codigo", "sexo", "ciudad_direccion_fisica", PAM_COL_LR]],
    on="codigo", how="inner"
).dropna()

print(f"Registros en df_p1:     {len(df_p1):,}")
print(f"Estudiantes únicos:     {df_p1['codigo'].nunique():,}")
print(f"Actividades únicas:     {df_p1['actividad'].nunique()}")
print("\nTop 10 actividades (por frecuencia):")
print(df_p1["actividad"].value_counts().head(10).to_string())


Registros en df_p1:     161,854
Estudiantes únicos:     17,907
Actividades únicas:     1000

Top 10 actividades (por frecuencia):
actividad
prestamo                          17578
caf                               10570
alimentacion                       8953
caf presencial                     7648
maraton de estudio                 6431
taller exito academico             6143
taller para el exito academico     5713
tutor                              4236
becario                            3334
transporte                         3291


In [30]:
# Regresión Logística: pipeline preprocesamiento + clasificador
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = df_p1[["sexo", "ciudad_direccion_fisica", PAM_COL_LR]]
y = df_p1["actividad"]

cat_cols_lr = ["sexo", "ciudad_direccion_fisica"]
num_cols_lr = [PAM_COL_LR]

preprocessor_lr = ColumnTransformer(
    transformers=[
        # drop='first' evita multicolinealidad perfecta;
        # handle_unknown='ignore' tolera ciudades no vistas en prueba.
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols_lr),
        # PAM se pasa sin transformar (numérica en escala comparable).
        ("num", "passthrough", num_cols_lr),
    ]
)

modelo_lr = Pipeline([
    ("preprocessing", preprocessor_lr),
    # max_iter=1000: el default de 100 no converge con muchas clases.
    ("logreg", LogisticRegression(max_iter=1000, random_state=42)),
])

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X, y, test_size=0.2, random_state=42
)
modelo_lr.fit(X_train_lr, y_train_lr)
y_pred_lr_cls = modelo_lr.predict(X_test_lr)

# ── Reporte de clasificación ──────────────────────────────────────────────────
print(classification_report(y_test_lr, y_pred_lr_cls, zero_division=0))

# ── Visualización: F1-score por actividad (top 15 por soporte) ────────────────
report_dict_lr = classification_report(
    y_test_lr, y_pred_lr_cls, output_dict=True, zero_division=0
)

metricas_lr = pd.DataFrame(
    {clase: v for clase, v in report_dict_lr.items()
     if clase not in ("accuracy", "macro avg", "weighted avg")
     and isinstance(v, dict)}
).T[["precision", "recall", "f1-score", "support"]]
metricas_lr = metricas_lr.sort_values("support", ascending=False).head(15)

wa_lr  = report_dict_lr.get("weighted avg", {})
acc_lr = report_dict_lr.get("accuracy", 0)

print(f"\nAccuracy global:      {acc_lr:.3f}")
print(f"F1 ponderado:         {wa_lr.get('f1-score', 0):.3f}")
print(f"Precision ponderada:  {wa_lr.get('precision', 0):.3f}")
print(f"Recall ponderado:     {wa_lr.get('recall', 0):.3f}")

colores_lr = [
    "#4A7FA5" if v >= 0.5 else "#E07B8B"
    for v in metricas_lr["f1-score"].values
]
fig_lr, ax_lr = plt.subplots(figsize=(10, 6))
bars_lr = ax_lr.barh(
    metricas_lr.index, metricas_lr["f1-score"].values,
    color=colores_lr, edgecolor="white"
)
for bar, val in zip(bars_lr, metricas_lr["f1-score"].values):
    ax_lr.text(
        val + 0.005, bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}", va="center", fontsize=9
    )
ax_lr.set_xlabel("F1-score")
ax_lr.set_xlim(0, 1.05)
ax_lr.axvline(0.5, color="gray", linewidth=0.8, linestyle="--", label="F1 = 0.5")
ax_lr.legend(fontsize=9)
ax_lr.set_title(
    f"F1-score por actividad — Regresión Logística (top 15 por soporte)\n"
    f"Accuracy = {acc_lr:.3f}  |  F1 ponderado = {wa_lr.get('f1-score', 0):.3f}",
    fontweight="bold"
)
plt.tight_layout()
guardar(fig_lr, "logreg_f1_actividades.png")
plt.show()

print("\nInterpretación:")
print("  · Azul  = F1 >= 0.5: el modelo discrimina bien esa actividad.")
print("  · Rojo  = F1 <  0.5: baja capacidad — posiblemente pocas muestras.")
print("  · La regresión logística es un baseline lineal; actividades con")
print("    distribuciones muy solapadas serán difíciles de separar sólo con")
print("    sexo, ciudad y PAM.")


KeyboardInterrupt: 